# Week 4 Day 4 — Missing Data Strategies

## What I built:
- Classified all missing columns as MCAR, MAR, or MNAR
- Compared 4 imputation strategies on LotFrontage
- Added missingness indicator columns
- Built reusable report_missing() function → src/utils.py

## MCAR / MAR / MNAR classification:
- MCAR: MasVnrArea, Electrical → random errors → simple impute
- MAR:  LotFrontage → depends on neighborhood → KNN impute
- MNAR: All Garage + Basement cols → missing = feature absent
        → create HasGarage, HasBasement indicator flags

## Imputation comparison (RMSE on val set):
Drop rows        : 0.1615  ← worst
SimpleImputer    : 0.1507
KNNImputer       : 0.1502
IterativeImputer : 0.1492  ← best

## Indicator column correlations with SalePrice:
HasMasVnr           : 0.3608 ← strongest
HasGarage           : 0.2368
HasBasement         : 0.1528
LotFrontage_missing : 0.0041 ← confirms MCAR

## report_missing() function:
Takes any DataFrame → returns missing analysis with
recommended strategy per column (DROP/KNN/SIMPLE)
Saved to src/utils.py for reuse in all future projects

# Classify missing columns as MCAR/MAR/MNAR:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, KFold

# load data
df = pd.read_csv('house_prices_train.csv')

# classify missing columns
missing_analysis = {
    'Column'       : [],
    'Missing_count': [],
    'Missing_pct'  : [],
    'Type'         : [],
    'Reason'       : []
}

classifications = {
    'LotFrontage'  : ('MAR',  'Depends on neighborhood and lot shape'),
    'GarageType'   : ('MNAR', 'Missing means no garage exists'),
    'GarageYrBlt'  : ('MNAR', 'Missing means no garage exists'),
    'GarageFinish' : ('MNAR', 'Missing means no garage exists'),
    'GarageQual'   : ('MNAR', 'Missing means no garage exists'),
    'GarageCond'   : ('MNAR', 'Missing means no garage exists'),
    'BsmtFinType2' : ('MNAR', 'Missing means no basement exists'),
    'BsmtExposure' : ('MNAR', 'Missing means no basement exists'),
    'BsmtFinType1' : ('MNAR', 'Missing means no basement exists'),
    'BsmtCond'     : ('MNAR', 'Missing means no basement exists'),
    'BsmtQual'     : ('MNAR', 'Missing means no basement exists'),
    'MasVnrArea'   : ('MCAR', 'Likely data entry error — random'),
    'Electrical'   : ('MCAR', 'Only 1 missing — likely random error'),
}

for col, (mtype, reason) in classifications.items():
    count = df[col].isnull().sum()
    missing_analysis['Column'].append(col)
    missing_analysis['Missing_count'].append(count)
    missing_analysis['Missing_pct'].append(round(count/len(df)*100, 1))
    missing_analysis['Type'].append(mtype)
    missing_analysis['Reason'].append(reason)

result_df = pd.DataFrame(missing_analysis)
print("=" * 70)
print("   MISSING DATA CLASSIFICATION")
print("=" * 70)
print(result_df.to_string(index=False))
print("=" * 70)
print(f"\nMCAR columns: {result_df[result_df.Type=='MCAR'].Column.tolist()}")
print(f"MAR columns : {result_df[result_df.Type=='MAR'].Column.tolist()}")
print(f"MNAR columns: {result_df[result_df.Type=='MNAR'].Column.tolist()}")

   MISSING DATA CLASSIFICATION
      Column  Missing_count  Missing_pct Type                                Reason
 LotFrontage            259         17.7  MAR Depends on neighborhood and lot shape
  GarageType             81          5.5 MNAR        Missing means no garage exists
 GarageYrBlt             81          5.5 MNAR        Missing means no garage exists
GarageFinish             81          5.5 MNAR        Missing means no garage exists
  GarageQual             81          5.5 MNAR        Missing means no garage exists
  GarageCond             81          5.5 MNAR        Missing means no garage exists
BsmtFinType2             38          2.6 MNAR      Missing means no basement exists
BsmtExposure             38          2.6 MNAR      Missing means no basement exists
BsmtFinType1             37          2.5 MNAR      Missing means no basement exists
    BsmtCond             37          2.5 MNAR      Missing means no basement exists
    BsmtQual             37          2.5 MNAR

# Compare 4 imputation strategies on LotFrontage:

In [2]:
from sklearn.model_selection import train_test_split

# use only numeric columns for this comparison
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'SalePrice']

X = df[num_cols].copy()
y = np.log1p(df['SalePrice'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

# strategy 1 — drop rows with missing LotFrontage
X_train_drop = X_train.dropna(subset=['LotFrontage'])
y_train_drop  = y_train[X_train_drop.index]
X_val_drop    = X_val.dropna(subset=['LotFrontage'])
y_val_drop    = y_val[X_val_drop.index]

# strategy 2 — SimpleImputer (median)
simple = SimpleImputer(strategy='median')
X_train_simple = simple.fit_transform(X_train)
X_val_simple   = simple.transform(X_val)

# strategy 3 — KNNImputer
knn = KNNImputer(n_neighbors=5)
X_train_knn = knn.fit_transform(X_train)
X_val_knn   = knn.transform(X_val)

# strategy 4 — IterativeImputer
from sklearn.experimental import enable_iterative_imputer
iterative = IterativeImputer(random_state=42, max_iter=10)
X_train_iter = iterative.fit_transform(X_train)
X_val_iter   = iterative.transform(X_val)

# train RF on each and compare RMSE
def get_rmse(X_tr, y_tr, X_v, y_v):
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    preds = rf.predict(X_v)
    return np.sqrt(np.mean((y_v - preds) ** 2))

print("=" * 45)
print("   IMPUTATION STRATEGY COMPARISON")
print("=" * 45)
print(f"  Drop rows     : {get_rmse(X_train_drop, y_train_drop, X_val_drop, y_val_drop):.4f}")
print(f"  SimpleImputer : {get_rmse(X_train_simple, y_train, X_val_simple, y_val):.4f}")
print(f"  KNNImputer    : {get_rmse(X_train_knn, y_train, X_val_knn, y_val):.4f}")
print(f"  IterativeImputer: {get_rmse(X_train_iter, y_train, X_val_iter, y_val):.4f}")
print("=" * 45)

   IMPUTATION STRATEGY COMPARISON
  Drop rows     : 0.1615
  SimpleImputer : 0.1507
  KNNImputer    : 0.1502
  IterativeImputer: 0.1492


# Add missingness indicator columns:

In [3]:
# add indicator columns for MNAR features
df_indicators = df.copy()

# garage indicators
df_indicators['HasGarage']   = (df['GarageType'].notna()).astype(int)
df_indicators['HasBasement'] = (df['BsmtQual'].notna()).astype(int)
df_indicators['HasMasVnr']   = (df['MasVnrArea'] > 0).astype(int)

# LotFrontage was missing indicator
df_indicators['LotFrontage_missing'] = df['LotFrontage'].isnull().astype(int)

print("Indicator columns added:")
print(f"  HasGarage            : {df_indicators['HasGarage'].sum()} houses have garage")
print(f"  HasBasement          : {df_indicators['HasBasement'].sum()} houses have basement")
print(f"  HasMasVnr            : {df_indicators['HasMasVnr'].sum()} houses have masonry veneer")
print(f"  LotFrontage_missing  : {df_indicators['LotFrontage_missing'].sum()} houses missing lot frontage")

# check survival rate correlation with indicators
print("\nCorrelation of indicators with SalePrice:")
indicators = ['HasGarage', 'HasBasement', 'HasMasVnr', 'LotFrontage_missing']
for col in indicators:
    corr = df_indicators[col].corr(df_indicators['SalePrice'])
    print(f"  {col:25} : {corr:.4f}")

Indicator columns added:
  HasGarage            : 1379 houses have garage
  HasBasement          : 1423 houses have basement
  HasMasVnr            : 591 houses have masonry veneer
  LotFrontage_missing  : 259 houses missing lot frontage

Correlation of indicators with SalePrice:
  HasGarage                 : 0.2368
  HasBasement               : 0.1528
  HasMasVnr                 : 0.3608
  LotFrontage_missing       : 0.0041


# Build report_missing() function:

In [4]:
def report_missing(df):
    """
    Returns a DataFrame showing missing value analysis for each column.
    Recommended strategy: drop >40%, KNN 5-40%, simple <5%
    """
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    result = pd.DataFrame({
        'column'   : missing.index,
        'missing'  : missing.values,
        'pct'      : (missing.values / len(df) * 100).round(2),
        'dtype'    : [df[col].dtype for col in missing.index],
        'strategy' : [
            'DROP'         if pct > 40
            else 'KNN'     if pct > 5
            else 'SIMPLE'
            for pct in (missing.values / len(df) * 100)
        ]
    })
    
    print("=" * 60)
    print("   MISSING DATA REPORT")
    print("=" * 60)
    print(result.to_string(index=False))
    print("=" * 60)
    print(f"\nSummary:")
    print(f"  DROP   : {(result.strategy=='DROP').sum()} columns")
    print(f"  KNN    : {(result.strategy=='KNN').sum()} columns")
    print(f"  SIMPLE : {(result.strategy=='SIMPLE').sum()} columns")
    print("=" * 60)
    
    return result

# test it
missing_report = report_missing(df)

   MISSING DATA REPORT
      column  missing   pct   dtype strategy
      PoolQC     1453 99.52  object     DROP
 MiscFeature     1406 96.30  object     DROP
       Alley     1369 93.77  object     DROP
       Fence     1179 80.75  object     DROP
  MasVnrType      872 59.73  object     DROP
 FireplaceQu      690 47.26  object     DROP
 LotFrontage      259 17.74 float64      KNN
  GarageType       81  5.55  object      KNN
 GarageYrBlt       81  5.55 float64      KNN
GarageFinish       81  5.55  object      KNN
  GarageQual       81  5.55  object      KNN
  GarageCond       81  5.55  object      KNN
BsmtFinType2       38  2.60  object   SIMPLE
BsmtExposure       38  2.60  object   SIMPLE
BsmtFinType1       37  2.53  object   SIMPLE
    BsmtCond       37  2.53  object   SIMPLE
    BsmtQual       37  2.53  object   SIMPLE
  MasVnrArea        8  0.55 float64   SIMPLE
  Electrical        1  0.07  object   SIMPLE

Summary:
  DROP   : 6 columns
  KNN    : 6 columns
  SIMPLE : 7 columns
